# Soil Depth

In [ ]:
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rasterio.mask import mask
from rasterio.windows import Window
import contextily as ctx  # For basemap tiles
import os

# Paths to the raster and shapefile
raster_path = r"\\gis.slu.se\gisdata\sgu\jorddjupsmodell\vector\epsg3006\2024-02-22\delivery\jorddjup_10x10m\jorddjup_10x10m.tif"
# Load the catchments that I am running it for
zip_catch = "catch_316.zip"

os.chdir("C:/Users/anlr0006/Repositories/top-down/Shapefiles")


# Load the shapefiles from the ZIP files
gdf_catch = gpd.read_file(f"zip://{zip_catch}")

os.chdir("C:/Users/anlr0006/Repositories/DOC_catchments")

output_folder = r"maps"  # Folder to save maps

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Step 1: Read the shapefile (example)
gdf = gdf_catch # .iloc[1:3]  # Adjust for the number of catchments you want to process

# Step 2: Prepare to collect results and failures
statistics_results = []
failed_ids = []

# Process each shape in the catchment dataframe
for idx, zone in gdf.iterrows():
    try:
        # Extract the geometry and bounds of the current shape
        zone_geom = zone['geometry']
        bounds = zone_geom.bounds  # (min_x, min_y, max_x, max_y)

        # Open the raster file and process the data
        with rasterio.open(raster_path) as src:
            # Ensure shapefile CRS matches raster CRS
            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)

            # Convert bounds to raster pixel coordinates
            col_start, row_start = ~src.transform * (bounds[0], bounds[1])  # min_x, min_y
            col_stop, row_stop = ~src.transform * (bounds[2], bounds[3])    # max_x, max_y

            # Ensure valid window bounds
            col_start, col_stop = sorted([int(max(0, min(src.width - 1, col_start))),
                                           int(max(0, min(src.width - 1, col_stop)))] )
            row_start, row_stop = sorted([int(max(0, min(src.height - 1, row_start))),
                                           int(max(0, min(src.height - 1, row_stop)))])
            
            # Define the window
            window = Window(col_start, row_start, col_stop - col_start, row_stop - row_start)

            # Read the raster data for the window
            raster_data = src.read(1, window=window)

            # Plot the raster window and shapefile overlay
            fig, ax = plt.subplots(figsize=(10, 10))
            
            # Plot the raster with adjusted opacity (alpha controls transparency)
            ax.imshow(raster_data, cmap='gray', extent=(bounds[0], bounds[2], bounds[3], bounds[1]), alpha=0.6)
            ax.set_title(f"Catchment ID: {zone['mvm_id']}")  # Adjust ID field as necessary

            # Overlay the current shape on the plot
            gpd.GeoSeries(zone_geom).plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2)
            
            # Add a basemap using OpenStreetMap or Stamen Terrain
            ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)  # OSM default basemap
            
            # Save the map as an image file
            map_filename = os.path.join(output_folder, f"{zone['mvm_id']}_map.png")
            plt.savefig(map_filename, dpi=300)
            plt.close()

            # Mask the raster data using the current geometry
            out_image, out_transform = mask(src, [zone_geom], crop=True)

            # Extract valid raster values and calculate statistics
            masked_values = out_image[0]
            masked_values = masked_values[masked_values != src.nodata]

            if len(masked_values) > 0:
                statistics_results.append({
                    'mvm_id': zone['mvm_id'],  # Adjust to your zone ID field
                    'mean': np.mean(masked_values),
                    'stddev': np.std(masked_values),
                    'min': np.min(masked_values),
                    'max': np.max(masked_values),
                    '25th_percentile': np.percentile(masked_values, 25),
                    '75th_percentile': np.percentile(masked_values, 75)
                })
            else:
                failed_ids.append(zone['mvm_id'])

    except Exception as e:
        # If an error occurs, log the failure ID and continue
        print(f"Failed to process {zone['mvm_id']}: {e}")
        failed_ids.append(zone['mvm_id'])

# Step 3: Save statistics to CSV
statistics_df = pd.DataFrame(statistics_results)
statistics_df.to_csv(os.path.join(output_folder, "soil_depth.csv"), index=False)

# Step 4: Log failed IDs to a text file
with open(os.path.join(output_folder, "failed.txt"), 'w') as f:
    for failed_id in failed_ids:
        f.write(f"{failed_id}\n")

print(f"Processing complete. Maps saved to {output_folder}, statistics saved to soil_depth.csv, and failed IDs saved to failed.txt.")


# NDVI

In [ ]:
import ee
import geemap
# Authenticate
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='ee-anna-lackner')
print(ee.String('Hello from the Earth Engine servers!').getInfo())

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_0JLhFqfSY1uiEaW?source=Init


Hello from the Earth Engine servers!


In [ ]:
import os
import geopandas as gpd

zip_catch = "catch_316.zip"

os.chdir("C:/Users/anlr0006/Repositories/top-down/Shapefiles")

# Load the shapefiles from the ZIP files
gdf_catch = gpd.read_file(f"zip://{zip_catch}")

gdf_catch.to_crs(epsg=4326, inplace=True)

In [ ]:
import ee
import pandas as pd
import geopandas as gpd
from datetime import datetime

# Initialize the Earth Engine API
ee.Initialize()

# Function to load the shapefile and convert it to a FeatureCollection
def geo_to_ee(gdf):
    features = []
    for idx, geometry in gdf.geometry.items():
        try:
            mvm_id = gdf.loc[idx, 'mvm_id']  # Extract mvm_id for this row
            
            # Check the geometry type and convert accordingly
            if geometry.geom_type == 'Polygon':
                ee_geom = ee.Geometry.Polygon(list(geometry.exterior.coords))
                features.append(ee.Feature(ee_geom).set('mvm_id', mvm_id))  # Add 'mvm_id' as property
            elif geometry.geom_type == 'MultiPolygon':
                polygons = [ee.Geometry.Polygon(list(poly.exterior.coords)) for poly in geometry.geoms]
                ee_geom = ee.Geometry.MultiPolygon(polygons)
                features.append(ee.Feature(ee_geom).set('mvm_id', mvm_id))  # Add 'mvm_id' as property
            elif geometry.geom_type == 'Point':
                ee_geom = ee.Geometry.Point([geometry.x, geometry.y])
                features.append(ee.Feature(ee_geom).set('mvm_id', mvm_id))  # Add 'mvm_id' as property
            else:
                print(f"Skipping unsupported geometry type: {geometry.geom_type}")
        except Exception as e:
            print(f"Error processing geometry {idx}: {e}")
    return ee.FeatureCollection(features)

# Load your shapefile (replace 'gdf_catch' with the actual GeoDataFrame name)
gdf = gdf_catch.iloc[2:3]  # Example of selecting the first two rows for demonstration

# Print the DataFrame to check its structure
print("GeoDataFrame structure:")
print(gdf)
print(gdf.crs)

# Convert GeoDataFrame to Earth Engine FeatureCollection
catchments = geo_to_ee(gdf)

# Define the Landsat 8 NDVI Composite dataset (8-day composite)
Landsat_NDVI = ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_8DAY_NDVI')

# Define the months and years for which you want to calculate the NDVI (January to December, 2020-2021)
months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
years = [2020, 2021]

# Function to generate the correct last day of the month (handling leap years)
def get_last_day_of_month(year, month):
    if month == "02":  # Special handling for February (Leap Year)
        if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)):  # Leap year check
            return f"{year}-{month}-29"  # Leap year (February 29)
        else:
            return f"{year}-{month}-28"  # Non-leap year (February 28)
    elif month in ["04", "06", "09", "11"]:  # 30-day months
        return f"{year}-{month}-30"
    else:  # 31-day months
        return f"{year}-{month}-31"

# Store results in a list
results = []

# Loop through each geometry (catchment) and calculate NDVI time series for each month and year
for idx, geometry in gdf.iterrows():
    try:
        mvm_id = geometry['mvm_id']
        print(f"Processing geometry with mvm_id: {mvm_id}")
        
        # Convert the geometry to an Earth Engine object
        shape = geo_to_ee(gdf.loc[[idx]])  # Use the current row for the geometry
        print(f"Shape for mvm_id {mvm_id} converted to Earth Engine FeatureCollection.")
        
        # Loop through each year and month to create the time series
        for year in years:
            for month in months:
                print(f"Processing month: {month} for mvm_id {mvm_id} in {year}")
                
                # Initialize the collection for the current month
                image_collection = None
                
                start_date = f"{year}-{month}-01"
                end_date = get_last_day_of_month(year, month)
                
                # Filter the Landsat NDVI data for the current month and year
                date_filter = ee.Filter.date(start_date, end_date)
                
                # Filter Landsat data by date and geometry bounds
                landsat_filtered = Landsat_NDVI \
                    .filter(date_filter) \
                    .filterBounds(shape)
                
                # Check how many images are available for the current month
                image_count = landsat_filtered.size().getInfo()
                print(f"Number of images for {month}-{year}: {image_count}")
                
                # If no images are available, skip to the next month
                if image_count == 0:
                    print(f"No images for {month}-{year} and mvm_id {mvm_id}. Skipping.")
                    continue

                # Apply the reducers separately: mean, median, and stdDev
                ndvi_mean = landsat_filtered.reduce(ee.Reducer.mean())
                ndvi_median = landsat_filtered.reduce(ee.Reducer.median())
                ndvi_stdDev = landsat_filtered.reduce(ee.Reducer.stdDev())

                # Now apply reduceRegions for each of the statistics
                stats_mean = ndvi_mean.reduceRegions(
                    collection=shape,
                    reducer=ee.Reducer.mean(),
                    scale=30,  # Set scale to 30 meters (Landsat resolution)
                    tileScale=2
                )

                stats_median = ndvi_median.reduceRegions(
                    collection=shape,
                    reducer=ee.Reducer.median(),
                    scale=30,
                    tileScale=2
                )

                stats_stdDev = ndvi_stdDev.reduceRegions(
                    collection=shape,
                    reducer=ee.Reducer.stdDev(),
                    scale=30,
                    tileScale=2
                )

                try:
                    # Get the results for each statistic (mean, median, stdDev)
                    mean_info = stats_mean.getInfo()
                    median_info = stats_median.getInfo()
                    stdDev_info = stats_stdDev.getInfo()

                    # Check if the results contain features
                    if 'features' in mean_info and len(mean_info['features']) > 0:
                        # Extract and append results for each mvm_id, month, and year
                        for i in range(len(mean_info['features'])):
                            feature_data = {
                                'month': month,
                                'year': year,
                                'mvm_id': mvm_id,
                                'NDVI_mean': mean_info['features'][i]['properties'].get('mean', None),
                                'NDVI_median': median_info['features'][i]['properties'].get('median', None),
                                'NDVI_stdDev': stdDev_info['features'][i]['properties'].get('stdDev', None),
                                'coordinates': mean_info['features'][i]['geometry']['coordinates']
                            }
                            results.append(feature_data)

                except Exception as e:
                    print(f"Error computing statistics for {month}-{year} and mvm_id {mvm_id}: {str(e)}")
                    # If error occurs, append None for NDVI statistics
                    feature_data = {
                        'month': month,
                        'year': year,
                        'mvm_id': mvm_id,
                        'NDVI_mean': None,
                        'NDVI_median': None,
                        'NDVI_stdDev': None,
                        'coordinates': None
                    }
                    results.append(feature_data)

    except Exception as e:
        print(f"Error processing mvm_id {gdf.loc[idx, 'mvm_id']}: {e}")

# Convert results to a pandas DataFrame
df = pd.DataFrame(results)

# Display the final DataFrame

out_path = r"C:\Users\anlr0006\Repositories\DOC_catchments\Output\NDVI.csv"

df.to_csv(out_path, index = False)


GeoDataFrame structure:
   mvm_id        lat       lon          AU_CD   Shape_Leng     Shape_Area  \
2  1815.0  6330290.0  473043.0  633181-142382  2780.197525  214447.493165   

                                            geometry  
2  POLYGON ((14.55448 57.11567, 14.5554 57.11509,...  
EPSG:4326
Processing geometry with mvm_id: 1815.0
Shape for mvm_id 1815.0 converted to Earth Engine FeatureCollection.
Processing month: 01 for mvm_id 1815.0 in 2020
Number of images for 01-2020: 4
Processing month: 02 for mvm_id 1815.0 in 2020
Number of images for 02-2020: 4
Processing month: 03 for mvm_id 1815.0 in 2020
Number of images for 03-2020: 4
Processing month: 04 for mvm_id 1815.0 in 2020
Number of images for 04-2020: 3
Processing month: 05 for mvm_id 1815.0 in 2020
Number of images for 05-2020: 3
Processing month: 06 for mvm_id 1815.0 in 2020
Number of images for 06-2020: 4
Processing month: 07 for mvm_id 1815.0 in 2020
Number of images for 07-2020: 4
Processing month: 08 for mvm_id 1815.0 

# PLC 8

Too difficult to get a handle on. 

# Peat Area

In [1]:
import os
import geopandas as gpd

zip_catch = "catch_316.zip"

os.chdir("C:/Users/anlr0006/Repositories/top-down/Shapefiles")

# Load the shapefiles from the ZIP files
gdf_catch = gpd.read_file(f"zip://{zip_catch}")

In [2]:
import os
import rasterio
from xml.etree import ElementTree as ET

# Specify your folder path
folder_path = r"\\gis.slu.se\gisdata\slu\Torvkarta_1_0\Klassad_torvkarta"

# List the files in the folder
files_in_folder = os.listdir(folder_path)

# Filter out .tif, .tif.ovr, .tif.xml files
tif_files = [f for f in files_in_folder if f.endswith('.tif')]
ovr_files = [f for f in files_in_folder if f.endswith('.tif.ovr')]
xml_files = [f for f in files_in_folder if f.endswith('.tif.xml')]

print(f"Found .tif files: {tif_files}")
print(f"Found .tif.ovr files: {ovr_files}")
print(f"Found .tif.xml files: {xml_files}")

# Load and inspect a .tif file using rasterio
if tif_files:
    tif_path = os.path.join(folder_path, tif_files[0])
    with rasterio.open(tif_path) as src:
        print(f"Opened {tif_path}")
        print(f"CRS: {src.crs}")
        print(f"Dimensions: {src.width} x {src.height}")
        print(f"Number of Bands: {src.count}")
        print(f"Data Type: {src.dtypes[0]}")

# Optionally, load and inspect a .xml file
if xml_files:
    xml_path = os.path.join(folder_path, xml_files[0])
    tree = ET.parse(xml_path)
    root = tree.getroot()
    print(f"Root XML tag: {root.tag}")
    print(f"XML attributes: {root.attrib}")

# Check for overviews in .tif file
if tif_files and ovr_files:
    ovr_path = os.path.join(folder_path, ovr_files[0])
    with rasterio.open(tif_path) as src:
        if src.overviews(1):  # Check for overviews of the first band
            print(f"Overviews found for {tif_path}: {src.overviews(1)}")
        else:
            print(f"No overviews found for {tif_path}")


Found .tif files: ['ClassifiedPeatMap.tif']
Found .tif.ovr files: ['ClassifiedPeatMap.tif.ovr']
Found .tif.xml files: ['ClassifiedPeatMap.tif.xml']
Opened \\gis.slu.se\gisdata\slu\Torvkarta_1_0\Klassad_torvkarta\ClassifiedPeatMap.tif
CRS: PROJCS["SWEREF99 TM",GEOGCS["SWEREF99",DATUM["SWEREF99",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6619"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4619"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3006"]]
Dimensions: 325000 x 770000
Number of Bands: 1
Data Type: uint8
Root XML tag: metadata
XML attributes: {'{http://www.w3.org/XML/1998/namespace}lang': 'sv'}
Overviews found for \\gis.slu.se\gis

In [3]:
mvm_ids = [165]

gdf_catch = gdf_catch.loc[gdf_catch['mvm_id'].isin(mvm_ids)]

In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import box
from rasterio.mask import mask

# Class names mapping for the raster values (0, 1, 2, 3, 4)
class_names = {
    0: "water",  # Water
    1: "mineral_soil",  # Mineral Soil
    2: "above_30_peat",  # Peat depth ≥30 cm
    3: "above_40_peat",  # Peat depth ≥40 cm
    4: "above_50_peat",  # Peat depth ≥50 cm
}

# Function to divide a geometry into smaller sub-geometries
def split_geometry(geometry, rows, cols):
    bounds = geometry.bounds
    minx, miny, maxx, maxy = bounds
    width = maxx - minx
    height = maxy - miny

    # Generate smaller bounding boxes
    sub_geometries = []
    for i in range(rows):
        for j in range(cols):
            x0 = minx + (i * width / rows)
            y0 = miny + (j * height / cols)
            x1 = minx + ((i + 1) * width / rows)
            y1 = miny + ((j + 1) * height / cols)
            sub_geometries.append(box(x0, y0, x1, y1))

    return sub_geometries

# Function to process each sub-geometry and compute area
def process_sub_geometry(src, geometry):
    # Mask the raster with the current sub-geometry's geometry
    out_image, out_transform = mask(src, [geometry], crop=True)
    data = out_image[0]

    # Create a dictionary to store the area for each class for the current sub-geometry
    area_by_class = {class_name: 0 for class_name in class_names.values()}

    # Calculate the area for each class (0 to 4) within the masked region
    for class_value, class_name in class_names.items():
        class_mask = data == class_value
        area_pixels = np.sum(class_mask)
        area_by_class[class_name] += area_pixels * (src.res[0] * src.res[1])  # pixel area in m²

    return area_by_class

# Function to handle processing with dynamic grid resizing
def process_catchment_with_retry(src, geometry, max_retries=3, initial_rows=3, initial_cols=3):
    retries = 0
    success = False
    while retries < max_retries and not success:
        try:
            # Split the catchment geometry into smaller sub-geometries (adjust rows and cols if needed)
            sub_geometries = split_geometry(geometry, rows=initial_rows, cols=initial_cols)

            area_by_class_total = {class_name: 0 for class_name in class_names.values()}
            block_count = 0

            for sub_geometry in sub_geometries:
                # Process each sub-geometry and calculate area
                print(f"Processing sub-geometry {block_count + 1}/{len(sub_geometries)}...")
                sub_area_by_class = process_sub_geometry(src, sub_geometry)

                # Update total areas
                for class_name in area_by_class_total:
                    area_by_class_total[class_name] += sub_area_by_class[class_name]

                block_count += 1

            print(f"Catchment: Processed {block_count} sub-geometries.")

            # Calculate total area
            total_area = sum(area_by_class_total.values())
            print(f"Catchment: Total area calculated: {total_area} m²")

            success = True  # If everything goes well, we mark success
            return area_by_class_total, total_area

        except Exception as e:
            print(f"Error processing with grid size {initial_rows}x{initial_cols}: {e}")
            retries += 1
            initial_rows *= 2  # Double the number of rows
            initial_cols *= 2  # Double the number of columns
            print(f"Retrying with larger grid: {initial_rows}x{initial_cols} (Attempt {retries}/{max_retries})")

    # If retries exceed max_retries, log and return None
    print(f"Failed to process catchment after {max_retries} attempts.")
    return None, None

# Initialize a list to store the results for each catchment
results = []

# Open the raster file
with rasterio.open(tif_path) as src:
    print(f"Opened raster file: {tif_path}")
    print(f"Raster size: {src.width}x{src.height}, resolution: {src.res}, dtype: {src.dtypes[0]}")

    # Loop through each catchment geometry in gdf_catch
    for idx, row in gdf_catch.iterrows():
        try:
            geometry = row['geometry']
            catchment_id = row['mvm_id']
            print(f"Processing catchment {catchment_id}: Geometry bounds: {geometry.bounds}")

            # Process catchment with retry logic
            area_by_class_total, total_area = process_catchment_with_retry(src, geometry)

            if area_by_class_total is not None:
                # Append results for this catchment
                results.append({
                    'mvm_id': catchment_id,
                    **area_by_class_total,
                    'total_area': total_area
                })
            else:
                print(f"Skipping catchment {catchment_id} due to repeated errors.")
        except Exception as e:
            print(f"Error processing catchment {row['mvm_id']}: {e}")
            continue

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Set the mvm_id column as the index of the DataFrame
results_df.set_index('mvm_id', inplace=True)

# Display the resulting table (DataFrame)
print("Processing complete. Results:")
print(results_df)

# Save results to a CSV file
os.chdir(r"C:\Users\anlr0006\Repositories\DOC_catchments\Output")
results_df.to_csv('peat_area_by_class.csv')
print("Results saved to peat_area_by_class.csv")


Opened raster file: \\gis.slu.se\gisdata\slu\Torvkarta_1_0\Klassad_torvkarta\ClassifiedPeatMap.tif
Raster size: 325000x770000, resolution: (2.0, 2.0), dtype: uint8
Processing catchment 165.0: Geometry bounds: (523837.4003999997, 7274784.5801, 674212.3307999996, 7421592.920000002)
Processing sub-geometry 1/9...
Processing sub-geometry 2/9...
